# Advanced RAG

The hybrid retrieval pipeline from [notebook 07](/courses/llm-eng/07-rag-pipeline.html) — ChromaDB dense search fused with BM25 via Reciprocal Rank Fusion, followed by cross-encoder reranking — represents the production standard at most financial services firms today. Yet a gap remains: the pipeline still treats the user's raw query as the retrieval signal, which introduces a systematic mismatch between question-space and document-space. This deep dive closes that gap with four techniques that attack the problem from different angles: **query rewriting** (LLM-generated query variants that increase lexical and semantic coverage), **HyDE** (Hypothetical Document Embeddings, which shifts retrieval from question-space into document-space by embedding a synthetic answer), **RAPTOR** (recursive abstractive processing that indexes summaries alongside raw chunks to handle multi-hop questions), and **multi-vector indexing** (representing each chunk at two granularities — summary and full text — so retrieval fires on the summary but returns the full chunk). We benchmark all four against the notebook 07 baseline on the same 20-document SEC filings corpus using Recall@5 and Mean Reciprocal Rank.

Setup:

In [ ]:
#| echo: false
import os, json
import numpy as np
from dotenv import load_dotenv
load_dotenv()

import openai
from pydantic import BaseModel
from typing import Optional, Type

PRICES = {
    "gpt-4o":      {"input": 2.50,  "output": 10.00},
    "gpt-4o-mini": {"input": 0.15,  "output": 0.60},
}

class LLMClient:
    def __init__(self, model="gpt-4o-mini", temperature=0.0):
        self.model = model; self.temperature = temperature
        self._client = openai.OpenAI()
        self._in = 0; self._out = 0

    def complete(self, messages, *, response_format=None):
        if response_format is not None:
            resp = self._client.beta.chat.completions.parse(
                model=self.model, messages=messages,
                temperature=self.temperature, response_format=response_format)
        else:
            resp = self._client.chat.completions.create(
                model=self.model, messages=messages, temperature=self.temperature)
        if resp.usage:
            self._in += resp.usage.prompt_tokens; self._out += resp.usage.completion_tokens
        if response_format is not None: return resp.choices[0].message.parsed
        return resp.choices[0].message.content

    @property
    def total_cost(self):
        if self.model not in PRICES: return 0.0
        p = PRICES[self.model]
        return (self._in * p["input"] + self._out * p["output"]) / 1_000_000

llm = LLMClient()

## The Query–Document Mismatch Problem

Every RAG failure mode traces back to a single root cause: [the retrieval query and the target document do not live in the same region of embedding space]{.mark}. This mismatch takes several forms.

**Vocabulary mismatch.** A user asks "how exposed is the firm to rate moves?" The relevant document says "Interest rate risk represents one of the most significant market risks." The phrase "rate moves" has low token overlap with "interest rate risk", and even the dense embedding may not bridge this gap reliably because the model has learned that "rate moves" is colloquial while "interest rate risk" is a formal SEC term used in a different distribution of surrounding context.

<br>

**Question-space vs. document-space.** Embedding a question produces a vector in a different region of space than embedding an answer, even if the two are semantically equivalent. This is not a defect in the embedding model — it reflects the genuine structural difference between interrogative and declarative text. A question like "What is the CET1 ratio?" has a very different structure from the passage "Our CET1 ratio was 14.8% at year-end." The cross-encoder in notebook 07 partially compensates for this by running bidirectional attention over the pair, but it still operates on the top-$k$ results from an initial retrieval step that inherits the mismatch.

<br>

**Granularity mismatch.** Multi-hop questions — "compare the firm's CET1 ratio to its capital return guidance for 2025" — require synthesizing information from multiple sections. No single chunk contains the answer, so single-vector retrieval always misses at least one piece. The techniques in this notebook address each of these failure modes: query rewriting attacks vocabulary mismatch; HyDE attacks question-space vs. document-space mismatch; RAPTOR addresses multi-hop granularity by building a hierarchy of summaries; and multi-vector indexing improves precision by allowing retrieval at the natural granularity of the question.

## Corpus and Baseline Infrastructure

We reuse the 20-document SEC filings corpus and the `ChromaStore`, `BM25Retriever`, and `CrossEncoderReranker` classes from [notebook 07](/courses/llm-eng/07-rag-pipeline.html). The corpus spans four sections — `risk_factors`, `mda`, `capital_liquidity`, and `guidance` — covering the topics most frequently targeted by financial analyst queries.

In [ ]:
CORPUS = [
    # Section: risk_factors (indices 0-4)
    {"id": "chunk-00", "text": "Interest rate risk represents one of the most significant market risks facing the firm. A 100 basis point increase in interest rates would reduce the fair value of our fixed-rate debt portfolio by approximately $2.3 billion.", "section": "risk_factors"},
    {"id": "chunk-01", "text": "Credit risk arises from the potential that a counterparty will fail to perform its obligations. We manage credit risk through diversification, collateral requirements, and credit limits by counterparty.", "section": "risk_factors"},
    {"id": "chunk-02", "text": "Operational risk includes the risk of loss resulting from inadequate or failed internal processes, people, systems, or external events, including cybersecurity threats and technology failures.", "section": "risk_factors"},
    {"id": "chunk-03", "text": "Our derivatives portfolio had a notional value of $1.2 trillion at year-end. Net market value exposure after netting and collateral was $18.4 billion, primarily concentrated in interest rate and foreign exchange derivatives.", "section": "risk_factors"},
    {"id": "chunk-04", "text": "Our Value at Risk (VaR) at the 99th percentile for a one-day holding period was $142 million, reflecting the diversified nature of our trading portfolios across equities, fixed income, and commodities.", "section": "risk_factors"},
    # Section: mda (indices 5-9)
    {"id": "chunk-05", "text": "Net revenues for the fiscal year were $47.4 billion, an increase of 8% compared to the prior year. The increase was driven primarily by higher net interest income reflecting the rising interest rate environment.", "section": "mda"},
    {"id": "chunk-06", "text": "Investment banking revenues decreased 23% to $6.1 billion, reflecting lower advisory fees amid reduced M&A activity and a challenging environment for equity and debt underwriting.", "section": "mda"},
    {"id": "chunk-07", "text": "Return on equity for the year was 12.4%, compared to 15.1% in the prior year. Book value per share increased to $312.50, up from $290.20.", "section": "mda"},
    {"id": "chunk-08", "text": "Net interest margin expanded 18 basis points to 2.94%, driven by higher short-term rates partially offset by increased funding costs. Provision for credit losses increased to $2.1 billion, reflecting normalization from historically low levels.", "section": "mda"},
    {"id": "chunk-09", "text": "Assets under management in our investment management segment grew to $2.8 trillion, an increase of 6% from prior year. Prime brokerage revenues increased 12% to $4.3 billion on higher client balances and margin loan activity.", "section": "mda"},
    # Section: capital_liquidity (indices 10-14)
    {"id": "chunk-10", "text": "Our Common Equity Tier 1 (CET1) capital ratio was 14.8% at year-end, well above the regulatory minimum of 4.5% and our internal target of 13%.", "section": "capital_liquidity"},
    {"id": "chunk-11", "text": "We maintain a liquidity coverage ratio (LCR) of 128%, exceeding the regulatory requirement of 100%. Our high-quality liquid assets totaled $280 billion at year-end.", "section": "capital_liquidity"},
    {"id": "chunk-12", "text": "Under Basel III framework requirements, our leverage ratio was 5.8%, comfortably above the 3% minimum. Our total risk-weighted assets were $1.47 trillion at year-end, consistent with prior year.", "section": "capital_liquidity"},
    {"id": "chunk-13", "text": "We are subject to FINRA Rule 4110 and SEC Rule 15c3-1 (the Net Capital Rule), which require us to maintain minimum net capital of not less than the greater of $250,000 or 2% of aggregate debit items. Our net capital exceeded the minimum by $12.4 billion.", "section": "capital_liquidity"},
    {"id": "chunk-14", "text": "The liquidity stress test results indicate the firm could withstand a 30-day severe market stress scenario while maintaining positive liquidity. The Internal Liquidity Adequacy Assessment Process (ILAAP) was completed in Q3 and reviewed by the Board Risk Committee.", "section": "capital_liquidity"},
    # Section: guidance (indices 15-19)
    {"id": "chunk-15", "text": "Looking ahead to fiscal 2025, management expects continued revenue growth in the range of 4-6%, supported by a stable rate environment and improving capital markets activity.", "section": "guidance"},
    {"id": "chunk-16", "text": "We plan to return $8 billion to shareholders through dividends and share repurchases in fiscal 2025, subject to regulatory approval and market conditions.", "section": "guidance"},
    {"id": "chunk-17", "text": "Earnings per share for fiscal 2024 were $42.30, compared to $47.20 in the prior year, reflecting lower net income partially offset by the reduction in diluted share count from ongoing repurchases.", "section": "guidance"},
    {"id": "chunk-18", "text": "Management has identified three strategic priorities for fiscal 2025: (1) expanding the wealth management client base to $500 billion in AUM, (2) growing transaction banking revenues by 15%, and (3) reducing the expense ratio below 65%.", "section": "guidance"},
    {"id": "chunk-19", "text": "We expect our CET1 ratio to remain in the 13.5-15.0% range through fiscal 2025, subject to regulatory stress test outcomes. Any excess capital above 14.5% will be returned to shareholders under our capital return policy.", "section": "guidance"},
]

print(f"Corpus: {len(CORPUS)} chunks")
for sec in ["risk_factors", "mda", "capital_liquidity", "guidance"]:
    n = sum(1 for c in CORPUS if c["section"] == sec)
    print(f"  {sec}: {n} chunks")

We rebuild the baseline retrieval infrastructure — `ChromaStore`, `BM25Retriever`, `CrossEncoderReranker`, and the `hybrid_retrieve` function — as a self-contained block:

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder


class ChromaStore:
    """ChromaDB vector store with metadata-filter support."""

    def __init__(self, persist_dir: str, collection: str):
        self._client = chromadb.PersistentClient(path=persist_dir)
        self._embed_fn = OpenAIEmbeddingFunction(
            api_key=os.environ["OPENAI_API_KEY"],
            model_name="text-embedding-3-small",
        )
        self._col = self._client.get_or_create_collection(
            name=collection, embedding_function=self._embed_fn
        )

    def reset(self, collection: str) -> None:
        """Drop and recreate the collection (idempotent re-index)."""
        try:
            self._client.delete_collection(collection)
        except Exception:
            pass
        self._col = self._client.get_or_create_collection(
            name=collection, embedding_function=self._embed_fn
        )

    def add_documents(
        self,
        texts: list[str],
        metadatas: list[dict] | None = None,
        ids: list[str] | None = None,
    ) -> None:
        n = len(texts)
        ids = ids or [f"doc-{self._col.count() + i}" for i in range(n)]
        metadatas = metadatas or [{} for _ in range(n)]
        self._col.add(documents=texts, metadatas=metadatas, ids=ids)

    def query(self, text: str, k: int = 5, where: dict | None = None) -> list[dict]:
        kwargs = {"query_texts": [text], "n_results": min(k, self._col.count())}
        if where:
            kwargs["where"] = where
        results = self._col.query(**kwargs)
        docs = results["documents"][0]
        metas = results["metadatas"][0]
        dists = results["distances"][0]
        return [{"text": d, "metadata": m, "score": 1 - dist}
                for d, m, dist in zip(docs, metas, dists)]

    def count(self) -> int:
        return self._col.count()


class BM25Retriever:
    """BM25 sparse retriever."""

    def __init__(self, corpus: list[dict]):
        self._corpus = corpus
        tokenized = [doc["text"].lower().split() for doc in corpus]
        self._bm25 = BM25Okapi(tokenized)

    def retrieve(self, query: str, k: int = 5) -> list[dict]:
        tokens = query.lower().split()
        scores = self._bm25.get_scores(tokens)
        ranked = np.argsort(scores)[::-1][:k]
        return [
            {"text": self._corpus[i]["text"],
             "metadata": {k: v for k, v in self._corpus[i].items() if k != "text"},
             "score": float(scores[i]), "rank": int(pos)}
            for pos, i in enumerate(ranked)
        ]


class CrossEncoderReranker:
    """Cross-encoder reranker."""

    def __init__(self, model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"):
        self._model = CrossEncoder(model_name)

    def rerank(self, query: str, candidates: list[dict], top_n: int = 5) -> list[dict]:
        pairs = [(query, c["text"]) for c in candidates]
        scores = self._model.predict(pairs)
        ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
        return [{**doc, "rerank_score": float(score)} for doc, score in ranked[:top_n]]


def hybrid_retrieve(
    query: str,
    store: ChromaStore,
    bm25: BM25Retriever,
    k: int = 5,
    rrf_k: int = 60,
    candidate_k: int = 20,
) -> list[dict]:
    """Dense + BM25 fused with Reciprocal Rank Fusion."""
    dense = store.query(query, k=candidate_k)
    sparse = bm25.retrieve(query, k=candidate_k)
    rrf_scores: dict[str, float] = {}
    doc_map: dict[str, dict] = {}
    for rank, doc in enumerate(dense, start=1):
        key = doc["text"]
        rrf_scores[key] = rrf_scores.get(key, 0.0) + 1.0 / (rrf_k + rank)
        doc_map[key] = doc
    for rank, doc in enumerate(sparse, start=1):
        key = doc["text"]
        rrf_scores[key] = rrf_scores.get(key, 0.0) + 1.0 / (rrf_k + rank)
        doc_map[key] = doc
    sorted_keys = sorted(rrf_scores, key=rrf_scores.__getitem__, reverse=True)[:k]
    return [{"text": key, "rrf_score": rrf_scores[key], **doc_map[key].get("metadata", {})}
            for key in sorted_keys]

Indexing the 20-chunk corpus and initializing the baseline components:

In [ ]:
PERSIST_DIR = "/tmp/chroma-dd01"
COLLECTION  = "filings-baseline"

store = ChromaStore(persist_dir=PERSIST_DIR, collection=COLLECTION)
store.reset(COLLECTION)
store.add_documents(
    texts=[c["text"] for c in CORPUS],
    metadatas=[{"section": c["section"], "id": c["id"]} for c in CORPUS],
    ids=[c["id"] for c in CORPUS],
)

bm25     = BM25Retriever(CORPUS)
reranker = CrossEncoderReranker()

print(f"Indexed {store.count()} chunks into ChromaDB.")

## Query Rewriting

**Query rewriting** addresses vocabulary mismatch by generating multiple paraphrases of the original query before retrieval. Each variant uses different wording, and we union the results across all variants before applying the cross-encoder. The intuition is simple: if the user asks "how exposed is the firm to rate moves?", one variant might produce "interest rate risk sensitivity" (matching the SEC risk-factors vocabulary), another might produce "fixed-rate portfolio fair value impact" (matching the specific passage about the \$2.3B impact), and a third might produce the original query.

We generate $n$ rewritten variants using the LLM, retrieve top-$k$ for each, take the union (deduplicated by text), then rerank the union with the cross-encoder. For a corpus of $C$ chunks this costs $n$ embedding calls but the total candidate set is at most $n \cdot k$ chunks before deduplication — well within the cross-encoder's throughput budget.

:::{.callout-note}
Query rewriting adds latency proportional to $n$ — one embedding call per variant. In practice $n = 3$ is the standard choice: two additional variants add meaningful coverage while keeping latency below 200ms for a cached embedding model.

:::

In [ ]:
from pydantic import BaseModel


class RewrittenQueries(BaseModel):  # <1>
    queries: list[str]


class QueryRewriter:
    """Generate n rewritten variants of a query for multi-perspective retrieval."""

    SYSTEM_PROMPT = (
        "You are a financial document retrieval assistant. "
        "Given a user question, generate {n} alternative phrasings that use "
        "different vocabulary and emphasis to improve recall over SEC filings. "
        "Include the original question as the first entry. "
        "Return ONLY a JSON object with a 'queries' key."
    )

    def __init__(self, llm_client: LLMClient, n: int = 3):
        self._llm = llm_client
        self._n = n

    def rewrite(self, query: str) -> list[str]:
        """Return n query variants including the original."""
        messages = [
            {"role": "system", "content": self.SYSTEM_PROMPT.format(n=self._n)},
            {"role": "user", "content": query},
        ]
        result = self._llm.complete(messages, response_format=RewrittenQueries)  # <2>
        return result.queries[:self._n]


rewriter = QueryRewriter(llm, n=3)

# Demonstrate on a vocabulary-mismatch query
q = "how exposed is the firm to rate moves?"
variants = rewriter.rewrite(q)
print(f"Original: {q}")
for i, v in enumerate(variants, 1):
    print(f"  Variant {i}: {v}")

1. We use a Pydantic model for structured output parsing, which guarantees the LLM returns exactly the fields we expect rather than free-form text that must be parsed with fragile string operations.
2. `response_format=RewrittenQueries` triggers the OpenAI structured output API: the model is constrained to emit valid JSON matching the schema. This is more reliable than asking for JSON in the system prompt.

The `QueryRewritingRetriever` wraps retrieval over all variants and deduplicates by text before reranking:

In [ ]:
class QueryRewritingRetriever:
    """Retrieve using multiple query variants, union results, then rerank."""

    def __init__(
        self,
        store: ChromaStore,
        bm25: BM25Retriever,
        reranker: CrossEncoderReranker,
        rewriter: QueryRewriter,
        k_per_query: int = 10,
        top_n: int = 5,
    ):
        self._store    = store
        self._bm25     = bm25
        self._reranker = reranker
        self._rewriter = rewriter
        self._k        = k_per_query
        self._top_n    = top_n

    def retrieve(self, query: str) -> list[dict]:
        variants  = self._rewriter.rewrite(query)  # <1>
        seen: dict[str, dict] = {}
        for variant in variants:
            for doc in hybrid_retrieve(variant, self._store, self._bm25, k=self._k):  # <2>
                seen.setdefault(doc["text"], doc)
        candidates = list(seen.values())  # <3>
        return self._reranker.rerank(query, candidates, top_n=self._top_n)  # <4>


qr_retriever = QueryRewritingRetriever(
    store=store, bm25=bm25, reranker=reranker, rewriter=rewriter,
    k_per_query=10, top_n=5,
)

results = qr_retriever.retrieve("how exposed is the firm to rate moves?")
print("Query-rewriting top-5:")
for i, r in enumerate(results, 1):
    print(f"  [{i}] rerank={r['rerank_score']:.3f} | {r['text'][:80]}...")

1. We generate $n$ variants of the original query (including the original itself as the first entry).
2. For each variant we run hybrid retrieval (dense + BM25 + RRF) and collect up to $k$ candidates. Using hybrid retrieval here rather than dense-only ensures that exact-match terms from the rewritten variants (e.g., "interest rate risk") are captured by BM25.
3. Results from all variants are merged into a dict keyed by text for $O(1)$ deduplication. A chunk retrieved by multiple variants still appears once in the candidate pool.
4. The cross-encoder scores every candidate against the **original** query, not the variants. Reranking on the original preserves fidelity to user intent.

## HyDE: Hypothetical Document Embeddings

**HyDE** was introduced by Gao et al. (2022) as a solution to the question-space vs. document-space gap. Rather than embedding the question and searching the document index, we ask the LLM to write a short hypothetical passage that *answers* the question, embed that passage, and use the passage embedding as the retrieval vector.

The key insight is geometric. Let $\mathbf{q} \in \mathbb{R}^d$ be the query embedding and $\mathbf{d}_i \in \mathbb{R}^d$ be the embedding of the $i$-th document chunk. The retrieval score is $\cos(\mathbf{q}, \mathbf{d}_i)$. Empirically, embedding models place declarative text (answers, statements) in a different manifold than interrogative text (questions). A hypothetical answer $\hat{a}$ lives in declarative space, so its embedding $\mathbf{e}(\hat{a})$ is geometrically closer to the document embeddings:

$$\cos(\mathbf{e}(\hat{a}),\, \mathbf{d}_i) \;\geq\; \cos(\mathbf{e}(q),\, \mathbf{d}_i)$$

even when $\hat{a}$ is factually incorrect — because the *style* and *vocabulary* of $\hat{a}$ already match the document distribution. The LLM does not need to know the right answer; it only needs to produce text that looks like the answer would look.

:::{.callout-caution}
HyDE adds one LLM call per query — roughly 50–150ms and \$0.00003 at `gpt-4o-mini` pricing. This is negligible for analyst tooling but may matter for high-throughput batch pipelines. Cache hypothetical answers by query hash to amortize the cost across repeated queries.

:::

In [ ]:
class HyDERetriever:
    """Retrieve by embedding a hypothetical answer to the query."""

    SYSTEM_PROMPT = (
        "You are a financial analyst writing internal research notes. "
        "Write a concise 2-3 sentence passage from an SEC filing that would "
        "directly answer the following question. Use formal financial language "
        "and plausible but invented numbers. Do not say you are generating a "
        "hypothetical document — just write the passage."
    )

    def __init__(
        self,
        store: ChromaStore,
        bm25: BM25Retriever,
        reranker: CrossEncoderReranker,
        llm_client: LLMClient,
        k: int = 20,
        top_n: int = 5,
    ):
        self._store    = store
        self._bm25     = bm25
        self._reranker = reranker
        self._llm      = llm_client
        self._k        = k
        self._top_n    = top_n

    def _generate_hypothesis(self, query: str) -> str:
        messages = [
            {"role": "system", "content": self.SYSTEM_PROMPT},
            {"role": "user", "content": query},
        ]
        return self._llm.complete(messages)  # <1>

    def retrieve(self, query: str) -> list[dict]:
        hypothesis = self._generate_hypothesis(query)  # <2>
        candidates = hybrid_retrieve(hypothesis, self._store, self._bm25, k=self._k)  # <3>
        return self._reranker.rerank(query, candidates, top_n=self._top_n)  # <4>


hyde = HyDERetriever(store=store, bm25=bm25, reranker=reranker, llm_client=llm, k=20, top_n=5)

# Demonstrate on a question where question-space mismatch is evident
q = "What is the firm's exposure to interest rate movements?"
hyp = hyde._generate_hypothesis(q)
print(f"Query:      {q}")
print(f"Hypothesis: {hyp}")

results = hyde.retrieve(q)
print("\nHyDE top-5:")
for i, r in enumerate(results, 1):
    print(f"  [{i}] rerank={r['rerank_score']:.3f} | {r['text'][:80]}...")

1. We use free-form LLM completion here rather than structured output — we want a natural paragraph, not a parsed object.
2. The hypothesis is generated unconditionally from the LLM's parametric knowledge. For a corpus the LLM has never seen, the hypothesis may contain incorrect numbers (e.g., it might generate "CET1 of 15.2%" when the actual value is 14.8%); this does not matter because the embedding model captures the *type* of content, not the exact values.
3. We use the hypothesis as the retrieval string for both the dense store and BM25. Dense retrieval benefits from the document-space embedding; BM25 benefits from the formal vocabulary the LLM generates ("interest rate risk", "basis points", "fair value").
4. Crucially, we rerank on the **original query** — not the hypothesis. We want the final ranking to reflect relevance to the user's intent, not relevance to the LLM's invented answer.

## RAPTOR: Recursive Abstractive Processing

**RAPTOR** (Sarthi et al., 2024) attacks granularity mismatch. When a user asks a multi-hop question — "compare the firm's current CET1 ratio to its 2025 capital guidance" — the answer requires synthesizing information from both the `capital_liquidity` and `guidance` sections. No single chunk contains the full answer, so single-vector retrieval always returns an incomplete candidate set.

RAPTOR's solution is to build a *hierarchy*: cluster the raw chunks, summarize each cluster into a higher-level passage, then index both the raw chunks and the summaries. At retrieval time, a multi-hop question may match a cluster summary even though no individual chunk matches well. We implement a simplified two-level hierarchy:

- **Level 0:** raw chunks (the 20 documents above)
- **Level 1:** one summary per section (4 summaries, one per `{risk_factors, mda, capital_liquidity, guidance}`)

The full RAPTOR algorithm uses Gaussian Mixture Model clustering over embedding vectors, which is section-agnostic. We use the section metadata as a proxy for the clustering step — appropriate here because our corpus is already segmented by section, and the section boundaries correspond exactly to the thematic clusters a GMM would discover.

In [ ]:
RAPTOR_SECTION_PROMPT = (
    "You are a financial analyst. Write a concise 3-4 sentence summary of the "
    "following excerpts from the {section} section of an SEC filing. "
    "Capture the most important numerical facts and risk disclosures. "
    "Excerpts:\n\n{texts}"
)


def build_raptor_summaries(corpus: list[dict], llm_client: LLMClient) -> list[dict]:
    """Generate one summary per section (Level 1 nodes)."""
    sections = sorted(set(c["section"] for c in corpus))
    summaries = []
    for sec in sections:
        chunks = [c for c in corpus if c["section"] == sec]  # <1>
        texts  = "\n\n".join(f"- {c['text']}" for c in chunks)
        prompt = RAPTOR_SECTION_PROMPT.format(section=sec, texts=texts)
        summary_text = llm_client.complete(
            [{"role": "user", "content": prompt}]  # <2>
        )
        summaries.append({
            "id":      f"summary-{sec}",
            "text":    summary_text,
            "section": sec,
            "level":   1,  # <3>
        })
        print(f"  [{sec}] → {summary_text[:90]}...")
    return summaries


print("Building RAPTOR Level-1 summaries:")
SUMMARIES = build_raptor_summaries(CORPUS, llm)

1. We group chunks by section — our proxy for the GMM clustering step. In production, replace this with `sklearn.mixture.GaussianMixture` over the chunk embeddings for corpus-agnostic hierarchical clustering.
2. We use a single user message here rather than a system/user split because the prompt is entirely instructional — there is no persistent persona needed across calls.
3. We tag each summary node with `level=1` (raw chunks are level 0) so we can filter by level during retrieval and inspection.

We index both raw chunks (level 0) and summaries (level 1) into a dedicated ChromaDB collection:

In [ ]:
RAPTOR_COLLECTION = "filings-raptor"

raptor_store = ChromaStore(persist_dir=PERSIST_DIR, collection=RAPTOR_COLLECTION)
raptor_store.reset(RAPTOR_COLLECTION)

# Index Level 0: raw chunks
raptor_store.add_documents(
    texts=[c["text"] for c in CORPUS],
    metadatas=[{"section": c["section"], "level": 0, "id": c["id"]} for c in CORPUS],
    ids=[c["id"] for c in CORPUS],
)

# Index Level 1: section summaries
raptor_store.add_documents(
    texts=[s["text"] for s in SUMMARIES],
    metadatas=[{"section": s["section"], "level": 1, "id": s["id"]} for s in SUMMARIES],
    ids=[s["id"] for s in SUMMARIES],
)

# BM25 over the combined corpus (raw + summaries)
raptor_bm25 = BM25Retriever(CORPUS + SUMMARIES)

print(f"RAPTOR store: {raptor_store.count()} documents (20 raw + 4 summaries)")

The RAPTOR retriever queries the combined index and, when a summary is returned, expands it back to its constituent raw chunks:

In [ ]:
class RaptorRetriever:
    """Two-level RAPTOR retriever: retrieves from combined raw+summary index."""

    def __init__(
        self,
        store: ChromaStore,
        bm25: BM25Retriever,
        reranker: CrossEncoderReranker,
        raw_corpus: list[dict],
        summaries: list[dict],
        k: int = 20,
        top_n: int = 5,
    ):
        self._store    = store
        self._bm25     = bm25
        self._reranker = reranker
        self._raw      = {c["id"]: c for c in raw_corpus}
        self._summary_sections = {s["section"]: s for s in summaries}
        self._k        = k
        self._top_n    = top_n

    def retrieve(self, query: str) -> list[dict]:
        candidates = hybrid_retrieve(query, self._store, self._bm25, k=self._k)  # <1>
        expanded: dict[str, dict] = {}
        for doc in candidates:
            meta = doc.get("metadata", doc)  # <2>
            if meta.get("level", 0) == 1:
                section = meta.get("section", "")
                for chunk in self._raw.values():
                    if chunk["section"] == section:  # <3>
                        expanded.setdefault(chunk["id"], {"text": chunk["text"]})
            else:
                expanded.setdefault(meta.get("id", doc["text"]), doc)
        return self._reranker.rerank(query, list(expanded.values()), top_n=self._top_n)  # <4>


raptor = RaptorRetriever(
    store=raptor_store, bm25=raptor_bm25, reranker=reranker,
    raw_corpus=CORPUS, summaries=SUMMARIES, k=20, top_n=5,
)

q = "Compare the firm's current capital ratios to its 2025 capital return guidance."
results = raptor.retrieve(q)
print(f"RAPTOR top-5 for: '{q}'")
for i, r in enumerate(results, 1):
    print(f"  [{i}] rerank={r['rerank_score']:.3f} | {r['text'][:80]}...")

1. We query the combined store (raw + summaries). A multi-hop query may fire on a summary because the summary explicitly contains the cross-section synthesis the question requires.
2. Metadata is stored inconsistently across paths (`store.query` returns `{"metadata": {...}}` while `hybrid_retrieve` flattens metadata to the top level). We handle both conventions.
3. When a level-1 summary is retrieved, we expand it to all raw chunks from that section. This trades some precision for recall on multi-hop questions: the relevant chunks are guaranteed to be in the candidate pool.
4. The cross-encoder prunes the expanded candidate set back to the top $n$ most relevant chunks, restoring precision.

## Multi-Vector Indexing

**Multi-vector indexing** decouples the retrieval signal from the returned content. The standard single-vector approach embeds the full chunk and retrieves by it: if the chunk is long, the embedding is a coarse average of all its content and may not represent any specific sub-claim precisely. Multi-vector indexing stores each chunk under two vectors: (1) a **summary embedding** (a short sentence that captures the chunk's main claim, optimized for retrieval) and (2) the **full chunk text** that is returned to the LLM. Retrieval fires on the compact summary embedding, which is a better representation of the retrievable claim, and the result returned to the LLM is the full passage with context.

This is the same idea as the "parent-child" or "small-to-big" chunking strategy: store small (summary) for retrieval, return large (full chunk) for generation. The difference here is that both the small and large representations come from the same source passage — we generate the summary with an LLM rather than using a hierarchical chunking scheme.

In [ ]:
SUMMARY_PROMPT = (
    "Summarize the following SEC filing excerpt in ONE sentence of at most 20 words. "
    "Preserve all specific numbers and identifiers.\n\nPassage: {text}"
)


def generate_chunk_summaries(corpus: list[dict], llm_client: LLMClient) -> list[dict]:
    """Generate a one-sentence summary for each chunk."""
    enriched = []
    for chunk in corpus:
        summary = llm_client.complete(
            [{"role": "user", "content": SUMMARY_PROMPT.format(text=chunk["text"])}]
        )
        enriched.append({**chunk, "summary": summary})
    return enriched


print("Generating per-chunk summaries for multi-vector indexing:")
CORPUS_WITH_SUMMARIES = generate_chunk_summaries(CORPUS, llm)

# Preview a few summary/full-text pairs
for chunk in CORPUS_WITH_SUMMARIES[:3]:
    print(f"\n  Full:    {chunk['text'][:80]}...")
    print(f"  Summary: {chunk['summary']}")

We index the one-sentence summaries as the retrieval vectors while storing the full chunk text in the metadata for return:

In [ ]:
MV_COLLECTION = "filings-multivec"

mv_store = ChromaStore(persist_dir=PERSIST_DIR, collection=MV_COLLECTION)
mv_store.reset(MV_COLLECTION)

mv_store.add_documents(
    texts=[c["summary"] for c in CORPUS_WITH_SUMMARIES],  # <1>
    metadatas=[
        {"full_text": c["text"], "section": c["section"], "id": c["id"]}  # <2>
        for c in CORPUS_WITH_SUMMARIES
    ],
    ids=[c["id"] for c in CORPUS_WITH_SUMMARIES],
)

print(f"Multi-vector store: {mv_store.count()} summary vectors indexed.")

# BM25 also on summaries for consistent hybrid retrieval
mv_bm25 = BM25Retriever([{"text": c["summary"], "section": c["section"], "id": c["id"]}
                          for c in CORPUS_WITH_SUMMARIES])

1. We embed the one-sentence summary, not the full chunk. The summary is a more precise retrieval signal because it distills the single most important claim rather than averaging over all content in a long passage.
2. The full chunk text is stored in the `full_text` metadata field. ChromaDB metadata supports string values up to ~512KB, which is sufficient for typical financial filing chunks.

The `MultiVectorRetriever` queries the summary index but returns the full chunk text to the LLM:

In [ ]:
class MultiVectorRetriever:
    """Retrieve by summary embedding, return full chunk text."""

    def __init__(
        self,
        store: ChromaStore,
        bm25: BM25Retriever,
        reranker: CrossEncoderReranker,
        k: int = 20,
        top_n: int = 5,
    ):
        self._store    = store
        self._bm25     = bm25
        self._reranker = reranker
        self._k        = k
        self._top_n    = top_n

    def retrieve(self, query: str) -> list[dict]:
        candidates = hybrid_retrieve(query, self._store, self._bm25, k=self._k)  # <1>
        # Replace summary text with full chunk text from metadata
        expanded = []
        for doc in candidates:
            full_text = doc.get("full_text") or doc.get("metadata", {}).get("full_text")
            if full_text:
                expanded.append({**doc, "text": full_text})  # <2>
            else:
                expanded.append(doc)
        return self._reranker.rerank(query, expanded, top_n=self._top_n)  # <3>


mv_retriever = MultiVectorRetriever(
    store=mv_store, bm25=mv_bm25, reranker=reranker, k=20, top_n=5
)

q = "What are the liquidity coverage ratio and HQLA balance?"
results = mv_retriever.retrieve(q)
print(f"Multi-vector top-5 for: '{q}'")
for i, r in enumerate(results, 1):
    print(f"  [{i}] rerank={r['rerank_score']:.3f} | {r['text'][:90]}...")

1. The query is run against the summary-indexed store. The embedding of the query is now compared to compact, claim-focused summaries rather than long verbose passages — the similarity signal is sharper.
2. Before reranking we swap the summary text for the full chunk text. The cross-encoder then scores the `(query, full_chunk)` pair, which gives it the full context needed to assess relevance accurately.
3. Reranking on full-text candidates after summary-based retrieval combines the precision of summary retrieval with the accuracy of full-text reranking.

## Benchmarking

We evaluate all four advanced techniques against the notebook 07 baseline using two metrics. **Recall@5** measures whether the ground-truth chunk appears anywhere in the top 5 results — a necessary condition for the LLM to answer correctly. **Mean Reciprocal Rank (MRR)** rewards systems that place the correct chunk higher in the ranking:

$$\text{MRR} = \frac{1}{|Q|} \sum_{q \in Q} \frac{1}{\text{rank}_q}$$

where $\text{rank}_q$ is the 1-based rank of the ground-truth chunk for query $q$ (and $\frac{1}{\infty} = 0$ if not retrieved). MRR is more informative than Recall@$k$ because it distinguishes a system that always returns the correct chunk at rank 1 from one that returns it at rank 5.

In [ ]:
# Ground truth: 15 queries each pinned to a single ground-truth chunk index
EVAL_SET = [
    ("What is the CET1 capital ratio?",                                    10),
    ("What is the liquidity coverage ratio and HQLA balance?",             11),
    ("How did investment banking revenues perform?",                        6),
    ("What is the net interest margin?",                                    8),
    ("What are the cybersecurity and technology risks?",                    2),
    ("What is the Basel III leverage ratio?",                              12),
    ("What is the VaR at the 99th percentile?",                            4),
    ("FINRA Rule 4110 Net Capital Rule compliance",                        13),
    ("What are earnings per share for fiscal 2024?",                       17),
    ("Assets under management and prime brokerage revenues?",               9),
    ("How exposed is the firm to interest rate movements?",                  0),
    ("What are the 2025 fiscal year revenue growth expectations?",          15),
    ("What is the capital return plan for shareholders in 2025?",           16),
    ("What are the strategic priorities for fiscal 2025?",                  18),
    ("What is the derivatives portfolio notional value and net exposure?",   3),
]


def evaluate_retriever(
    retrieve_fn,
    eval_set: list[tuple[str, int]],
    corpus: list[dict],
    k: int = 5,
) -> dict:
    """Compute Recall@k and MRR for a retriever function."""
    recall_hits = 0
    reciprocal_ranks = []
    for query, gt_idx in eval_set:
        gt_text  = corpus[gt_idx]["text"]
        results  = retrieve_fn(query)
        texts    = [r["text"] for r in results[:k]]
        if gt_text in texts:
            rank = texts.index(gt_text) + 1
            recall_hits += 1
            reciprocal_ranks.append(1.0 / rank)
        else:
            reciprocal_ranks.append(0.0)
    return {
        f"recall@{k}": recall_hits / len(eval_set),
        "mrr":         np.mean(reciprocal_ranks),
    }

We define the five retrieval pipelines to compare. The baseline is hybrid + rerank from notebook 07; the remaining four add one advanced technique each:

In [ ]:
def baseline_retrieve(q):
    candidates = hybrid_retrieve(q, store, bm25, k=20)
    return reranker.rerank(q, candidates, top_n=5)


PIPELINES = {
    "baseline (nb07)": baseline_retrieve,
    "+ query rewriting": qr_retriever.retrieve,
    "+ HyDE":           hyde.retrieve,
    "+ RAPTOR":         raptor.retrieve,
    "+ multi-vector":   mv_retriever.retrieve,
}

print("Running benchmark (15 queries × 5 pipelines)...")
RESULTS = {}
for name, fn in PIPELINES.items():
    RESULTS[name] = evaluate_retriever(fn, EVAL_SET, CORPUS, k=5)
    print(f"  {name:<25} done")

Printing the results table:

In [ ]:
#| code-fold: true
header = f"{'Pipeline':<26} {'Recall@5':>10} {'MRR':>8}"
print(header)
print("-" * len(header))
for name, metrics in RESULTS.items():
    r = metrics["recall@5"]
    m = metrics["mrr"]
    flag = " ◀" if name != "baseline (nb07)" and r > RESULTS["baseline (nb07)"]["recall@5"] else ""
    print(f"{name:<26} {r:>10.3f} {m:>8.3f}{flag}")

print(f"\nTotal LLM cost so far: ${llm.total_cost:.4f}")

**Figure.** Each row is a complete retrieval pipeline evaluated on the 15-query SEC filings benchmark. Recall@5 is the fraction of queries for which the ground-truth chunk appears in the top 5 results. MRR weights by rank position — a system that always returns the correct chunk at rank 1 scores MRR = 1.0; one that returns it at rank 5 scores MRR = 0.20. Rows marked with ◀ improve on the baseline.

A few patterns to observe: query rewriting gives the largest Recall@5 gain for vocabulary-mismatch queries (e.g., "how exposed is the firm to rate moves?" → correctly retrieves chunk 0). HyDE improves MRR more than Recall — it retrieves the correct chunk but also tends to surface it higher in the ranking. RAPTOR's advantage is most visible on multi-hop queries that require synthesizing across sections. Multi-vector indexing improves precision on long-tail queries where the full chunk's embedding is diluted by peripheral context.

:::{.callout-important}
No single technique dominates across all query types. In production, the practical choice is to combine all four: query rewriting + HyDE for the retrieval step, multi-vector indexing as the index structure, and RAPTOR summaries as an additional index layer for multi-hop coverage. The combined pipeline adds roughly 3–4 LLM calls per query and 2× the index storage — a worthwhile tradeoff for an analyst assistant where retrieval quality directly affects decision quality.

:::

## Appendix: Ensemble Retrieval

It is natural to ask whether we can combine all four techniques simultaneously rather than choosing one. We can — and for production systems this is usually the right answer. The ensemble uses query rewriting to generate query variants, runs HyDE alongside them to generate a document-space query, retrieves from both the RAPTOR combined index and the multi-vector index, unions all candidates, and reranks.

The cost is approximately $n + 1$ LLM calls per query (one for each rewritten variant plus one for the hypothesis), plus the cross-encoder pass over the expanded candidate pool. For $n = 3$ variants and $k = 10$ per variant this yields at most $3 \times 10 + 10 = 40$ candidates before deduplication — still fast for the cross-encoder.

In [ ]:
class EnsembleRetriever:
    """Combined pipeline: query rewriting + HyDE + RAPTOR index + multi-vector index."""

    def __init__(
        self,
        base_store: ChromaStore,
        base_bm25: BM25Retriever,
        raptor_store: ChromaStore,
        raptor_bm25: BM25Retriever,
        mv_store: ChromaStore,
        mv_bm25: BM25Retriever,
        raw_corpus: list[dict],
        summaries: list[dict],
        reranker: CrossEncoderReranker,
        rewriter: QueryRewriter,
        llm_client: LLMClient,
        k_per_query: int = 10,
        top_n: int = 5,
    ):
        self._base_store    = base_store
        self._base_bm25     = base_bm25
        self._raptor_store  = raptor_store
        self._raptor_bm25   = raptor_bm25
        self._mv_store      = mv_store
        self._mv_bm25       = mv_bm25
        self._raw           = {c["id"]: c for c in raw_corpus}
        self._summary_map   = {s["section"]: s for s in summaries}
        self._reranker      = reranker
        self._rewriter      = rewriter
        self._llm           = llm_client
        self._k             = k_per_query
        self._top_n         = top_n

    def _hyde_query(self, query: str) -> str:
        return self._llm.complete([
            {"role": "system", "content": HyDERetriever.SYSTEM_PROMPT},
            {"role": "user",   "content": query},
        ])

    def retrieve(self, query: str) -> list[dict]:
        variants   = self._rewriter.rewrite(query)  # <1>
        hypothesis = self._hyde_query(query)         # <2>
        search_queries = variants + [hypothesis]

        seen: dict[str, dict] = {}
        for q in search_queries:
            # From RAPTOR index (raw + summaries)
            for doc in hybrid_retrieve(q, self._raptor_store, self._raptor_bm25, k=self._k):
                meta = doc.get("metadata", doc)
                if meta.get("level", 0) == 1:  # <3>
                    for chunk in self._raw.values():
                        if chunk["section"] == meta.get("section", ""):
                            seen.setdefault(chunk["id"], {"text": chunk["text"]})
                else:
                    seen.setdefault(meta.get("id", doc["text"]), doc)
            # From multi-vector index
            for doc in hybrid_retrieve(q, self._mv_store, self._mv_bm25, k=self._k):
                full = doc.get("full_text") or doc.get("metadata", {}).get("full_text")
                key  = doc.get("id") or doc.get("metadata", {}).get("id", doc["text"])
                if full:
                    seen.setdefault(key, {**doc, "text": full})  # <4>
                else:
                    seen.setdefault(key, doc)

        return self._reranker.rerank(query, list(seen.values()), top_n=self._top_n)  # <5>


ensemble = EnsembleRetriever(
    base_store=store,     base_bm25=bm25,
    raptor_store=raptor_store, raptor_bm25=raptor_bm25,
    mv_store=mv_store,    mv_bm25=mv_bm25,
    raw_corpus=CORPUS,    summaries=SUMMARIES,
    reranker=reranker,    rewriter=rewriter,
    llm_client=llm,       k_per_query=10, top_n=5,
)

q = "Compare the firm's current capital ratios to its 2025 capital return guidance."
results = ensemble.retrieve(q)
print(f"Ensemble top-5 for: '{q}'")
for i, r in enumerate(results, 1):
    print(f"  [{i}] rerank={r['rerank_score']:.3f} | {r['text'][:90]}...")

1. Query rewriting generates $n = 3$ variants (including the original), providing lexical diversity over the retrieval queries.
2. The HyDE hypothesis is generated once and added as a fourth retrieval query. This covers document-space matching that the question-phrased variants may miss.
3. Level-1 RAPTOR summary hits are expanded to their constituent chunks — the same expansion logic as in `RaptorRetriever`.
4. Multi-vector hits are swapped to full-text before insertion into the seen set, ensuring the cross-encoder has the complete passage to score.
5. The cross-encoder scores all deduplicated candidates against the **original query**, not any variant or hypothesis, preserving fidelity to the user's intent.

Benchmarking the ensemble against the individual techniques:

In [ ]:
#| code-fold: true
ensemble_metrics = evaluate_retriever(ensemble.retrieve, EVAL_SET, CORPUS, k=5)
all_results = {**RESULTS, "ensemble (all)": ensemble_metrics}

baseline_r5 = RESULTS["baseline (nb07)"]["recall@5"]
header = f"{'Pipeline':<26} {'Recall@5':>10} {'MRR':>8} {'Δ Recall@5':>12}"
print(header)
print("-" * len(header))
for name, metrics in all_results.items():
    r    = metrics["recall@5"]
    m    = metrics["mrr"]
    delta = r - baseline_r5
    sign  = "+" if delta >= 0 else ""
    print(f"{name:<26} {r:>10.3f} {m:>8.3f} {sign}{delta:>11.3f}")

print(f"\nTotal LLM cost so far: ${llm.total_cost:.4f}")

:::{.callout-note}
The ensemble benchmark triggers multiple LLM and embedding API calls per query — approximately 4 LLM calls and 4–5 embedding calls per query × 15 queries. In a notebook context with real API keys this takes 30–60 seconds and costs roughly \$0.002. If you are running without API keys, the LLM calls will fail; replace `llm.complete(...)` in `QueryRewriter` and `HyDERetriever` with mock responses that return fixed strings to keep the benchmark runnable.

:::

## Exercises

1. **Ablate HyDE temperature.** The `LLMClient` defaults to `temperature=0.0`, which produces the same hypothesis on every call. Modify `HyDERetriever` to generate $k = 3$ hypotheses at `temperature=0.7` and retrieve for each, then union and rerank. Measure whether the higher-variance hypotheses improve Recall@5 on vocabulary-mismatch queries.

2. **Implement GMM clustering for RAPTOR.** Replace the section-based clustering in `build_raptor_summaries` with a proper clustering step: embed all 20 chunks, fit a `GaussianMixture(n_components=4)` from `sklearn.mixture`, assign each chunk to its most likely cluster, and generate one summary per cluster. Compare the cluster-based RAPTOR summaries to the section-based ones on the 15-query benchmark.

3. **Build a latency profiler.** Wrap each retrieval pipeline in a timing decorator that records the wall-clock latency per query. After running the benchmark, print a table of `{pipeline: mean_latency_ms}`. Identify the dominant cost driver for each pipeline (embedding API round-trip, LLM call, or cross-encoder inference) and propose one concrete optimization for the slowest pipeline.

---

$\blacksquare$